In [ ]:
!nvidia-smi -L || echo "no GPU — BEiT3 sẽ fail ở startup"

# ── Tầng 1: bootstrap — chỉ 2 keys để clone được repo (AppConfig chưa tồn tại) ──
import os
from pathlib import Path

runtime = Path("/kaggle/working/boldsearch-runtime")
runtime.mkdir(exist_ok=True)

env_file = next((e / ".env" for e in Path("/kaggle/input").iterdir() if (e / ".env").is_file()), None)
assert env_file, "thiếu env dataset — attach dataset private của CI vào Input"

for line in env_file.read_text(encoding="utf-8").splitlines():
    key, sep, value = line.strip().partition("=")
    if sep and key in {"REPO_URL", "GH_PAT"}:
        os.environ.setdefault(key, value.strip().strip('"'))

# askpass ghi bằng python: IPython sẽ expand $GH_PAT nếu dùng !printf
askpass = Path("/tmp/boldsearch-askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GH_PAT" ;; esac\n')
askpass.chmod(0o700)
os.environ["GIT_ASKPASS"] = str(askpass)
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!if [ -d BoldSearch/.git ]; then git -C BoldSearch fetch --depth 1 origin main && git -C BoldSearch reset --hard FETCH_HEAD; else rm -rf BoldSearch && git clone --depth 1 "$REPO_URL" BoldSearch; fi
!rm -f /tmp/boldsearch-askpass.sh
!cp {env_file} BoldSearch/app/backend/.env
!git -C BoldSearch rev-parse --short HEAD

# ── Tầng 2: AppConfig — nguồn env chính từ đây ──
import sys

sys.path.insert(0, "/kaggle/working/BoldSearch/app/backend")
from app_config import app_config  # noqa: E402 — đọc .env vừa copy, validate toàn bộ config

# deploy-only key: cloudflared đọc token từ env TUNNEL_TOKEN, không qua argv (ps)
os.environ["TUNNEL_TOKEN"] = app_config.DEPLOY_KAGGLE.CF_TUNNEL_TOKEN
print("AppConfig OK —", app_config.SYSTEM_NAME, "| KEYFRAMES_DIR:", app_config.KEYFRAMES_DIR)

In [ ]:
!pip install -q uv
!uv sync --frozen --directory /kaggle/working/BoldSearch/app/backend
!echo "env ready: /kaggle/working/BoldSearch/app/backend/.venv"

In [ ]:
# (tuỳ chọn) BE public cho FE local gọi — cần DEPLOY_KAGGLE__CF_TUNNEL_TOKEN/HOSTNAME trong .env
cloudflared = runtime / "cloudflared"
!test -x {cloudflared} || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {cloudflared} && chmod +x {cloudflared})
!test -z "$TUNNEL_TOKEN" || (pkill -f 'cloudflared tunnel' || true; nohup {cloudflared} tunnel --no-autoupdate run > {runtime}/tunnel.log 2>&1 &)
!test -z "$TUNNEL_TOKEN" || (for i in $(seq 1 30); do grep -q 'Registered tunnel connection' {runtime}/tunnel.log 2>/dev/null && break; sleep 2; done; grep -m1 'Registered tunnel connection' {runtime}/tunnel.log || (echo 'TUNNEL FAIL'; tail -20 {runtime}/tunnel.log; false))
print('tunnel skip (chưa cấu hình token)' if not os.environ.get('TUNNEL_TOKEN') else 'tunnel OK')

# (tuỳ chọn) start BE ở nền
!cd BoldSearch/app/backend && nohup uv run uvicorn main:app --host 127.0.0.1 --port 8000 > /kaggle/working/uvicorn.log 2>&1 &
!sleep 8; tail -5 /kaggle/working/uvicorn.log
print(f"BE public: https://{app_config.DEPLOY_KAGGLE.CF_TUNNEL_HOSTNAME} -> 127.0.0.1:8000" if os.environ.get('TUNNEL_TOKEN') else "BE: 127.0.0.1:8000 (chưa có tunnel)")
print("BOOTSTRAP DONE")

In [ ]:
# giữ container sống cho CI-pushed run — attach editor/VS Code vào session này
!sleep infinity